In [1]:
import pandas as pd


In [2]:
df = pd.read_excel("CorporateSalary.xlsx")
df.head(2)


,Designation,Education,Department,TenureYears,ProjectsCompleted,PerformanceScore,Salary
0,Senior Analyst,Bachelor's,Data Science,8.7,12,4.6,119300
1,Director,Master's,Marketing,6.9,13,4.3,212200


In [3]:
# Split the data in to X and y
y = df[['Salary']]
print(y.head(2))
X = df.drop('Salary',axis=1)
X.head(2)


   Salary
0  119300
1  212200


,Designation,Education,Department,TenureYears,ProjectsCompleted,PerformanceScore
0,Senior Analyst,Bachelor's,Data Science,8.7,12,4.6
1,Director,Master's,Marketing,6.9,13,4.3


# Perform Train and test split




In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train_true, y_test_true = train_test_split(X,y,test_size=0.2,random_state=42)


In [5]:

OneHot_columns = ['Department']

ordinal_columns = ['Designation','Education']
numberical_cols = ['TenureYears','ProjectsCompleted','PerformanceScore']

Designation_Ord = ['Junior Analyst','Senior Analyst', 'Lead Consultant','Manager', 'Senior Manager','Director' ]
Education_Ord = ["Bachelor's", "Master's", 'PhD']



In [6]:
# Encoding AND Escaling

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler

# OrdinalEncoding_01 = (Name_Cat,OrdinalEncoder(categories=[Col_1_Order, Col_2_Order]),Ordinal_Col_Name,)
# OneHotEncoding_01 =  (NameOFcAT,OneHotEncoder(sparse_output=False, drop="first", handle_unknown="ignore"),OneHot_Col_Name,)
# Scaling = ("num", StandardScaler(), Scaling_Col_Name)

OrdinalEncoding_01 = ("Orinal_Cols",OrdinalEncoder(categories=[Designation_Ord, Education_Ord]),ordinal_columns,)
OneHotEncoding_01 =  ("One_Hot_Cols",OneHotEncoder(sparse_output=False, drop="first", handle_unknown="ignore"),OneHot_columns,)

Scaling = ("Scaling_01", StandardScaler(), numberical_cols)

transform_engine = ColumnTransformer(transformers=[OrdinalEncoding_01,OneHotEncoding_01,Scaling]).set_output(transform="pandas")



In [7]:
df.head(2)


,Designation,Education,Department,TenureYears,ProjectsCompleted,PerformanceScore,Salary
0,Senior Analyst,Bachelor's,Data Science,8.7,12,4.6,119300
1,Director,Master's,Marketing,6.9,13,4.3,212200


In [8]:
X_train_Trans = transform_engine.fit_transform(X_train)
X_train_Trans


,Orinal_Cols__Designation,Orinal_Cols__Education,One_Hot_Cols__Department_Engineering,One_Hot_Cols__Department_HR,One_Hot_Cols__Department_Marketing,One_Hot_Cols__Department_Product,One_Hot_Cols__Department_Sales,Scaling_01__TenureYears,Scaling_01__ProjectsCompleted,Scaling_01__PerformanceScore
42,0.0,0.0,0.0,1.0,0.0,0.0,0.0,-0.058633,1.084800,0.063193
12,3.0,2.0,0.0,0.0,0.0,1.0,0.0,-1.049288,-0.200889,-1.030770
15,0.0,2.0,0.0,0.0,0.0,0.0,1.0,0.324121,-0.299788,-0.135709
114,2.0,1.0,0.0,0.0,0.0,1.0,0.0,-1.274436,-1.090981,-1.726929
76,3.0,1.0,1.0,0.0,0.0,0.0,0.0,0.504240,0.095809,0.958254
...,...,...,...,...,...,...,...,...,...,...
106,0.0,0.0,1.0,0.0,0.0,0.0,0.0,-1.274436,2.667186,1.256608
14,0.0,1.0,0.0,1.0,0.0,0.0,0.0,-1.116832,0.689203,0.162644
92,3.0,2.0,0.0,0.0,1.0,0.0,0.0,0.571784,0.491405,0.958254
51,3.0,0.0,0.0,0.0,0.0,1.0,0.0,-0.036118,0.689203,1.157156


In [37]:
# Save Tranfomer in to pickle File
import joblib
joblib.dump(transform_engine,"Transform/transform.pkl")


['Transform/transform.pkl']

In [10]:
# pip3 install -U scikit-learn
# pip install sklearn


In [33]:
# Trainig Phase

from sklearn.linear_model import LinearRegression

linearmodel = LinearRegression()

linearmodel.fit(X_train_Trans,y_train_true)

joblib.dump(linearmodel,"Transform\linearModel.pkl")





['Transform\\linearModel.pkl']

In [13]:
y_train_true


,Salary
42,78100
12,146000
15,98500
114,60000
76,167400
...,...
106,40000
14,76900
92,168900
51,149700


In [14]:

y_train_pred = linearmodel.predict(X_train_Trans)



In [15]:
import numpy as np
# calcualte Error
# MSE
# MAE
# RMSE


from sklearn import metrics
mse = metrics.mean_squared_error(y_train_true,y_train_pred)
mae = metrics.mean_absolute_error(y_train_true,y_train_pred)
rmse = np.sqrt(mse)

print('Training')
print("Train_MSE:", mse)
print("Train_MAE:",mae)
print("Train_RMSE:",mae)


Training
Train_MSE: 1198962328.511402
Train_MAE: 21823.753534366075
Train_RMSE: 21823.753534366075


In [16]:
#  r2, ar2 % how good is the model 0 to 1
# 0 bad model
# 1 then good
# 0.85 0.87 0.89.

r2_score = metrics.r2_score(y_train_true,y_train_pred)
new_R2 = float(r2_score)

new_R2


0.7679795007422043

In [17]:
import numpy as np
from sklearn.metrics import r2_score

n = len(y_train_pred)  # number of observations
p = len(y_train_pred[0]) if len(y_train_true.shape) > 1 else 1  # number of features
adj_r2 = 1 - (1 - new_R2) * (n - 1) / (n - p - 1)


adj_r2


0.7655111975586107

In [30]:
from sklearn.metrics import mean_absolute_error, mean_squared_error,r2_score

def metrics(Actual,Predicted):
    MAE = mean_absolute_error(Actual,Predicted) 
    MSE = mean_squared_error(Actual,Predicted)
    RMSE = np.sqrt(MSE)
    R2_sCORE = r2_score(Actual,Predicted)

    n = len(y_train_pred)  # number of observations
    p = len(y_train_pred[0]) if len(y_train_true.shape) > 1 else 1  # number of features
    adj_r2 = 1 - (1 - new_R2) * (n - 1) / (n - p - 1)



    newDict = { 'MAE': round(MAE,3),            
                'MSE': round(MSE,3),
                'RMSE': round(RMSE,3),
                'R2_sCORE':round(R2_sCORE,3),
                'adj_r2':round(adj_r2,3) }
    
    return newDict


In [31]:
metrics(y_train_true,y_train_pred)



{'MAE': 21823.754,
 'MSE': 1198962328.511,
 'RMSE': 34626.035,
 'R2_sCORE': 0.768,
 'adj_r2': 0.766}

In [ ]:
# Tesing Phase


In [ ]:
import joblib

transformer_obj = joblib.load("\Transform\transform.pkl")
X_test_transform = transformer_obj.transform(X_test)

X_test_transform


OSError: [Errno 22] Invalid argument: 'Transform\transform.pkl'

In [ ]:
Liner_Model = joblib.load("linearModel.pkl")

y_test_pred = Liner_Model.predict(X_test_transform)

y_test_pred


array([[ 91995.62955567],
       [157869.83077474],
       [ 89592.65853228],
       [207657.15932862],
       [ 71234.97840307],
       [ 98194.43380761],
       [213938.35241162],
       [ 84263.14013551],
       [100223.39631616],
       [474962.43324436],
       [140454.00187496],
       [193640.34269979],
       [178977.86800187],
       [129619.72440055],
       [118277.06906176],
       [171134.6259243 ],
       [223268.7566815 ],
       [138671.01092544],
       [210560.80208355],
       [381921.24683434],
       [151422.80832474],
       [122531.69906883],
       [106854.18627042],
       [171973.00150037]])

In [ ]:
y_test_true


,Salary
44,93400
47,126400
4,90100
55,182800
26,93200
64,96800
73,163400
10,71300
40,94100
107,600000


In [ ]:
from sklearn import metrics
mse = metrics.mean_squared_error(y_test_true,y_test_pred)
mae = metrics.mean_absolute_error(y_test_true,y_test_pred)

rmse = np.sqrt(mse)

print('Training')
print("Train_MSE:", mse)
print("Train_MAE:",mae)
print("Train_RMSE:",mae)




Training
Train_MSE: 2573996111.819696
Train_MAE: 32797.16723566203
Train_RMSE: 32797.16723566203


In [ ]:
r2_score = metrics.r2_score(y_test_true,y_test_pred)
print(r2_score)





0.8485408112870985


In [ ]:
# n = Number of test observations
n = X_test.shape[0]

# p = Number of predictor variables (features)
p = X_test.shape[1]

# Adjusted R²
adj_r2 = 1 - (1 - r2_score) * (n - 1) / (n - p - 1)

print("Adjusted R²:", adj_r2)



Adjusted R²: 0.7950846270354861
